# Position Split and Position Merge

In [15]:
import csv
from web3 import Web3
from eth_abi import decode
import pandas as pd
from tqdm import tqdm

# Define the event signatures
event_signatures = {
    'PositionsMerge': 'PositionsMerge(address,address,bytes32,bytes32,uint256[],uint256)',
    'PositionsMergeSimplified': 'PositionsMerge(address,bytes32,uint256)',
    'PositionSplit': 'PositionSplit(address,address,bytes32,bytes32,uint256[],uint256)',
    'PositionSplitSimplified': 'PositionSplit(address,bytes32,uint256)',
}
# Compute the keccak hash of the event signatures to get the topic0
event_topic0 = {name: Web3.keccak(text=signature).hex() for name, signature in event_signatures.items()}
print(event_topic0)


{'PositionsMerge': '6f13ca62553fcc2bcd2372180a43949c1e4cebba603901ede2f4e14f36b282ca', 'PositionsMergeSimplified': 'ba33ac50d8894676597e6e35dc09cff59854708b642cd069d21eb9c7ca072a04', 'PositionSplit': '2e6bb91f8cbcda0c93623c54d0403a43514fabc40084ec96b6d5379a74786298', 'PositionSplitSimplified': 'bbed930dbfb7907ae2d60ddf78345610214f26419a0128df39b6cc3d9e5df9b0'}


In [42]:
print('Reading logs...')

def decode_address(topic):
    return '0x' + topic[-40:]

def decode_uint256(data):
    return int(data, 16)

def decode_position_split(log):
    decoded_event = {}
    decoded_event['event'] = 'PositionSplit'
    decoded_event['blockNumber'] = decode_uint256(log['blockNumber'])
    decoded_event['txIndex'] = decode_uint256(log['txIndex'])

    # Decode topics
    decoded_event['stakeholder'] = decode_address(log['topic1'])
    decoded_event['parentCollectionId'] = '0x' + log['topic2'][-64:]
    decoded_event['conditionId'] = '0x' + log['topic3'][-64:]

    # Decode data
    data = log['data']
    assert data.startswith('0x')
    data_hex = data[2:]
    data_bytes = bytes.fromhex(data_hex)

    #decode the rest
    decoded_head = decode(['address', 'uint256', 'uint256'], data_bytes[:96])
    collateralToken = decoded_head[0]
    partition_offset = decoded_head[1]
    amount = decoded_head[2]

    decoded_event['collateralToken'] = collateralToken
    decoded_event['amount'] = amount / 10e6

    # Decode the partition array
    partition_length = decode(['uint256'], data_bytes[partition_offset:partition_offset+32])[0]
    partition_data_bytes = data_bytes[partition_offset+32:partition_offset+32+32*partition_length]
    partition = decode(['uint256'] * partition_length, partition_data_bytes)
    decoded_event['partition'] = partition

    return decoded_event

# Initialize variables
total_log_entries = 0
decoded_events = []
non_decoded_events = []
non_decoded_events_log_index = []

# Set the range for log files
start = 50
end = 61

for i in tqdm(range(start, end), colour='green'):
    log_entries = []
    with open(f'ctf_exchange_neg_risk/logs_{i}M.csv', 'r') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            log_entries.append(row)

    for log in log_entries:
        topic0 = log['topic0']
        total_log_entries += 1
        if topic0 == '0x' + event_topic0['PositionSplit']:
            decoded_event = decode_position_split(log)
            decoded_events.append(decoded_event)

print('Decoded events:', len(decoded_events))
df = pd.DataFrame(decoded_events)
df.to_csv(f'positionsplit_events/positionsplit_{start}-{end}.csv', index=False)


Reading logs...


100%|██████████| 11/11 [03:01<00:00, 16.51s/it]


Decoded events: 820498


In [43]:
print('Reading logs...')

def decode_address(topic):
    return '0x' + topic[-40:]

def decode_uint256(data):
    return int(data, 16)

def decode_positions_merge(log):
    decoded_event = {}
    decoded_event['event'] = 'PositionsMerge'
    decoded_event['blockNumber'] = decode_uint256(log['blockNumber'])
    decoded_event['txIndex'] = decode_uint256(log['txIndex'])

    # Decode topics
    decoded_event['stakeholder'] = decode_address(log['topic1'])
    # decoded_event['topic1_stakeholder'] = log['topic1']
    decoded_event['collateralToken'] = decode_address(log['topic2']) 
    # decoded_event['topic2_parentCollectionId'] = log['topic2']
    decoded_event['parentCollectionId'] = '0x' + log['topic2'][-64:]
    # decoded_event['topic3_conditionId'] = log['topic3']
    decoded_event['conditionId'] = '0x' + log['topic3'][-64:]

    # Decode data
    data = log['data']
    # decoded_event['data'] = data
    
    assert data.startswith('0x')
    data_hex = data[2:]

    # Decode the partition array
    decoded_event['collateralToken'] = data_hex[24:64]     # Extracts the 20-byte token address
    # decoded_event['partition1'] = decode_uint256(data_hex[320:384])        # Partition 1
    # decoded_event['partition2'] = decode_uint256(data_hex[384:448])        # Partition 2
    amount = decode_uint256(data_hex[128:192])
    decoded_event['amount'] = amount / 10e6


    return decoded_event

# Initialize variables
total_log_entries = 0
decoded_events = []
non_decoded_events = []
non_decoded_events_log_index = []

# Set the range for log files
start = 50
end = 61

for i in tqdm(range(start, end), colour='green'):
    log_entries = []
    with open(f'ctf_exchange_neg_risk/logs_{i}M.csv', 'r') as csvfile:
        reader = csv.DictReader(csvfile)
        for row in reader:
            log_entries.append(row)

    for log in log_entries:
        topic0 = log['topic0']
        total_log_entries += 1
        if topic0 == '0x' + event_topic0['PositionsMerge']:
            decoded_event = decode_positions_merge(log)
            decoded_events.append(decoded_event)

print('Decoded events:', len(decoded_events))
df = pd.DataFrame(decoded_events)
df.to_csv(f'positionsmerge_events/positionsmerge_{start}-{end}.csv', index=False)

Reading logs...


100%|██████████| 11/11 [02:19<00:00, 12.71s/it]


Decoded events: 280705
